In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import statistics
import matplotlib.pyplot as plt
sns.set(font_scale=1.5)

Type="Quantile"
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df.pkl")

In [ ]:
# Step 1: Load the necessary Python libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import stats

# Step 2: Load the VIX price data and news sentiment data
vix_data = pd.DataFrame()
news_data = pd.DataFrame()

# Step 3: Calculate the VIX returns and news sentiment returns
vix_data['returns'] = df["VIX return"]#np.log(vix_data['Price']/vix_data['Price'].shift(1))
news_data['returns'] = df["Sentiment return"]#np.log(news_data['Sentiment']/news_data['Sentiment'].shift(1))


# Step 4: Define the event window and estimation window
event_window = 5  # Number of days in event window
estimation_window = 20  # Number of days in estimation window

# Step 5: Calculate the average VIX returns during the event window and the estimation window
vix_data['avg_returns_event'] = vix_data['returns'].rolling(window=event_window, min_periods=1).mean()
vix_data['avg_returns_estimation'] = vix_data['returns'].rolling(window=estimation_window, min_periods=1).mean()

# Step 6: Calculate the abnormal returns by subtracting the average returns during the estimation window
# from the average returns during the event window
vix_data['abnormal_returns'] = vix_data['avg_returns_event'] - vix_data['avg_returns_estimation']

# Step 7: Perform additional analysis and draw conclusions
mean_abnormal_returns = vix_data['abnormal_returns'].mean()
std_abnormal_returns = vix_data['abnormal_returns'].std()

# Calculate t-statistic and p-value for the null hypothesis of zero mean abnormal returns
t_statistic = mean_abnormal_returns / (std_abnormal_returns / np.sqrt(len(vix_data)))
p_value = 2 * (1 - abs(stats.t.cdf(t_statistic, df=len(vix_data)-1)))
vix_data.reset_index(inplace=True)

# Print the results
print("Event Study Results:")
print("Mean Abnormal Returns: {:.4f}".format(mean_abnormal_returns))
print("Standard Deviation of Abnormal Returns: {:.4f}".format(std_abnormal_returns))
print("t-statistic: {:.4f}".format(t_statistic))
print("p-value: {:.4f}".format(p_value))

# Plot the abnormal returns
fig, ax = plt.subplots()
ax.plot(vix_data['Date'], vix_data['abnormal_returns'], label='Abnormal Returns')
ax.axhline(0, color='black', linestyle='--')
ax.legend()
ax.set_xlabel('Date')
ax.set_ylabel('Abnormal Returns')
ax.set_title('Event Study of VIX Abnormal Returns Based on News Sentiment')
plt.show()